In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import linear_kernel
import joblib
import faiss
from sklearn.feature_extraction.text import TfidfTransformer

ModuleNotFoundError: No module named 'faiss'

In [8]:
print(faiss.__version__)

NameError: name 'faiss' is not defined

In [ ]:
basics = pd.read_csv("title.basics.tsv", sep="\t")
ratings = pd.read_csv("title.ratings.tsv", sep="\t")

In [ ]:
basics.head()

In [ ]:
ratings.head()

In [ ]:
# Combine both datasets on ratings
full_set = basics.merge(ratings, on="tconst", how="left")


In [ ]:
def breakdown(data):
    data_frames = {}
    for item in data.titleType.unique():
        data_frames[item] = data[data["titleType"] == item]
    return data_frames

In [ ]:
old_types = breakdown(full_set)

In [ ]:
old_types.keys()

In [ ]:
def counts_per_type(vals):
    numbers = {}
    for element in vals:
        numbers[element] = len(vals[element])
    return numbers

In [ ]:
print(counts_per_type(old_types))

In [ ]:
len(full_set)

In [ ]:
# number of null values per dataset

def nulls_per_type(elements):
    results = {}
    for element in elements:
        results[element] = elements[element].isnull().sum()
    return results

In [ ]:
print(nulls_per_type(old_types))

In [ ]:
#get the number of types without ratings vs the number of ratings
def has_ratings(elements):
    results = {}
    for element in elements:
        results[element] = {'len' : len(elements[element]), 'noRatings': int(elements[element]['averageRating'].isnull().sum()), 'noVotes' : int(elements[element]['numVotes'].isnull().sum())}
    return results

In [ ]:
has_ratings(old_types)

In [ ]:
full_set.dtypes

In [ ]:
basics.isnull().sum()

In [ ]:
#to avoid losing the genre in the one hot encoded version, make the genre blank, we can use the rating and dataset values to continue using it effectively
full_set["genres"] = full_set["genres"].fillna("")

In [ ]:
#drop the other null values without primary titles - its hard to recommend a movie without a title

full_set = full_set.dropna(subset=["primaryTitle", "originalTitle"])


In [ ]:
#one hot encoding of all genres from the dataset by separating all genres from strings and using get dummies  to clean them up.and one hot encode them.
with_genres = (
    full_set["genres"]
    .str.get_dummies(sep=",")
    .set_axis(full_set.index)
)

In [ ]:
full_set = pd.concat([full_set, with_genres], axis=1)

In [ ]:
print(full_set.index.equals(with_genres.index))

In [ ]:
full_set.columns

In [ ]:
full_set = full_set.loc[:, full_set.columns != "\\N"]

In [ ]:
full_set.dtypes

In [ ]:
def plot_balance(classes, counts):
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.bar(classes, counts)
    ax.bar_label(bars, fmt='{:,.0f}', padding=0.3)
    ax.set_title("Class Distribution")
    ax.set_xlabel("Type")
    ax.set_ylabel("Count")
    ax.spines['top'].set_visible(False)
    fig.autofmt_xdate()
    plt.tight_layout()

In [ ]:
def plot_class_distribution(data): 
    counter = has_ratings(data)
    
    # 1. Sort the items by the 'len' value inside the sub-dictionary
    sorted_items = sorted(counter.items(), key=lambda item: item[1]['len'], reverse=True)
    
    # 2. Extract the sorted keys and sorted lengths
    cls = [item[0] for item in sorted_items]
    cts = [item[1]['len'] for item in sorted_items]

    plot_balance(cls, cts)

In [ ]:
plot_class_distribution(old_types)

In [ ]:
# 1. Clean Numeric Columns (Handling the "\N" or strings in IMDb data)
full_set['startYear'] = pd.to_numeric(full_set['startYear'], errors='coerce')
full_set['averageRating'] = pd.to_numeric(full_set['averageRating'], errors='coerce')


In [ ]:
# Fill NaNs so the scaler doesn't crash (using median is safer than 0)
full_set['startYear'] = full_set['startYear'].fillna(full_set['startYear'].median())
full_set['averageRating'] = full_set['averageRating'].fillna(full_set['averageRating'].median())

# 2. Apply Log Transform to Votes
full_set['log_numVotes'] = np.log1p(full_set['numVotes'].fillna(0))

# 3. Scaling all numerical features to [0, 1]
scaler = MinMaxScaler()
# We scale Year, Log Votes, and Average Rating
num_features = ['startYear', 'log_numVotes', 'averageRating']
full_set[['scaled_year', 'scaled_votes', 'scaled_rating']] = scaler.fit_transform(full_set[num_features])

In [ ]:
# 4. Feature Selection & Weighting
# Optional: Multiply the numerical features by a weight (e.g., 0.5) 
# if you want Genres to be more important than Year/Rating.

genre_cols = full_set.columns[full_set.columns.get_loc("Action"):]
weight = 1.0 
full_set['scaled_year'] *= weight
full_set['scaled_votes'] *= weight
full_set['scaled_rating'] *= weight

# 5. Final Inference Matrix
feature_cols = list(genre_cols) + ['scaled_year', 'scaled_votes', 'scaled_rating']
inference_matrix = full_set[feature_cols].values

In [ ]:
# Filter for only what you need
keep_types = ['movie', 'tvSeries', 'short', 'tvMovie', 'tvMiniSeries', 'tvShort']
filtered_set = full_set[full_set["titleType"].isin(keep_types)]

In [ ]:
types = breakdown(filtered_set)

In [ ]:
plot_class_distribution(types)

In [ ]:
def plot_features(data): 
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(data, bins=30)
    ax.title("Feature Distribution")
    ax.xlabel("Value")
    ax.ylabel("Frequency")
    fig.autofmt_xdate()
    plt.tight_layout()

In [ ]:
def plot_genre_distribution(df):
    # 1. Identify the genre columns (binary columns from Action to Western)
    genre_cols = df.columns[df.columns.get_loc("Action"):]
    
    # 2. Sum the counts for each genre and sort ascending
    genre_counts = df[genre_cols].sum().sort_values(ascending=True)
    
    # 3. Plotting
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Use barh (horizontal bar) for better readability of many genres
    ax.barh(genre_counts.index, genre_counts.values, color='#3F888F', edgecolor='black')
    
    ax.set_title("Distribution of Movie Genres", fontsize=15)
    ax.set_xlabel("Number of Movies")
    ax.set_ylabel("Genre")
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_genre_distribution(full_set)

In [ ]:
def train_faiss_optimized(df, feature_columns):
    print("Step 1: TF-IDF Transformation...")
    tfidf = TfidfTransformer()
    # Stay in sparse format as long as possible
    sparse_matrix = tfidf.fit_transform(df[feature_columns])
    
    # Step 2: Initialize FAISS Index
    d = sparse_matrix.shape[1]
    # IndexFlatIP is great for Cosine Similarity
    index = faiss.IndexFlatIP(d)
    
    # Step 3: Add to Index in Batches 
    # This prevents your 16GB RAM from maxing out during the .toarray() conversion
    batch_size = 100000 
    total_rows = sparse_matrix.shape[0]
    
    print(f"Step 4: Adding {total_rows} rows to FAISS index in batches...")
    for i in range(0, total_rows, batch_size):
        end = min(i + batch_size, total_rows)
        # Convert only a small slice to dense at a time
        batch_dense = sparse_matrix[i:end].toarray().astype('float32')
        
        # Normalize each batch for Cosine Similarity
        faiss.normalize_L2(batch_dense)
        index.add(batch_dense)
        
        if i % 1000000 == 0:
            print(f"Processed {i} rows...")

    # Step 5: Save Assets
    print("Step 6: Saving model assets...")
    faiss.write_index(index, "movie_faiss.index")
    joblib.dump(indices, "indices_map.pkl")
    
    # Only save the metadata you actually need to show in the report
    meta_cols = ['primaryTitle', 'startYear', 'genres', 'averageRating']
    joblib.dump(df[meta_cols], "metadata.pkl")
    
    print("Done! Ready for inference.")
    return index

In [ ]:
index = train_faiss_optimized(filtered_set, feature_cols)

In [ ]:
def fast_inference(title, top_n=10):
    # Load assets
    idx_map = joblib.load("indices_map.pkl")
    meta = joblib.load("metadata.pkl")
    index = faiss.read_index("movie_recommender.index")
    
    # Get the vector for the input movie
    movie_idx = idx_map[title]
    # We reconstruct the vector from the index
    query_vector = index.reconstruct(int(movie_idx)).reshape(1, -1)
    
    # Search
    distances, indices = index.search(query_vector, top_n + 1)
    
    # Result
    return meta.iloc[indices[0][1:]] # Skip first result (self)

In [ ]:
# Do this once when starting your app/script
index = faiss.read_index("movie_faiss.index")
indices_map = joblib.load("indices_map.pkl")
metadata = joblib.load("metadata.pkl")

def get_recommendations_faiss(title, top_n=10):
    # 1. Clean and Verify Title
    title = title.strip()
    if title not in indices_map:
        return f"Error: '{title}' not found in index."

    # 2. Retrieve the Index Position
    # This is the integer position of the movie in the original dataframe
    movie_idx = indices_map[title]

    # 3. Reconstruct the Vector from the Index
    # Since we used IndexFlatIP, we can pull the normalized vector back out
    # Note: index.reconstruct returns a 1D array; search() needs 2D (1, d)
    query_vector = index.reconstruct(int(movie_idx)).reshape(1, -1)

    # 4. Search the Index
    # D = Cosine Similarity Scores (since we normalized L2)
    # I = Indices of the closest matches
    D, I = index.search(query_vector, top_n + 1)

    # 5. Process Results
    # Flatten the result and skip the first item (it's the movie itself)
    recommended_indices = I[0][1:]
    similarity_scores = D[0][1:]

    # 6. Build the Output Table
    results = metadata.iloc[recommended_indices].copy()
    results['match_score'] = [f"{round(s * 100, 2)}%" for s in similarity_scores]

    return results[['primaryTitle', 'startYear', 'genres', 'averageRating', 'match_score']]


In [ ]:
print(get_recommendations_faiss("The Matrix"))